# SNN 论文级可视化（Kaggle）

在 **同一 Session** 中：先 git clone 本仓库，把 **`best_msf_res2net.pt`** 放到 `/kaggle/working/`，并 **Add Data** PlantVillage。修改下方路径后 **Run All**。

生成（均基于 **验证集真实图像** + 训练好权重的一次前向，非 AI 臆造）：
- **`paper_snn_layer_time_heatmap.png`** — **层×时间步** 平均脉冲强度热力图（体现随 $t$ 的「波」在深度上传播）
- **`paper_snn_spike_raster.png`** — 各层 **子采样单元** 的 **Raster 点阵**（$x$ 轴为 $t$，点为发放事件）
- **`paper_snn_panel.png`** — first-spike 输入帧 + 首层 MSF 膜电位
- **`paper_snn_layer_spike_rates.png`** — 各脉冲层平均发放率柱状图
- `visualize_sample` 分项 PNG

**注意**：`num_classes`、权重与 checkpoint 一致；下一格设 **`BACKBONE = "sj_resnet18"`** 可加载 ResNet-18 SNN checkpoint（否则默认 **`"res2net"`**）。

In [ ]:
!pip -q install "spikingjelly>=0.0.0.0.14" matplotlib

In [ ]:
import os
import sys
from pathlib import Path

GITHUB_URL = "https://github.com/xing11234/plantvillage_snn.git"
BRANCH = "main"
CODE = Path("/kaggle/working/plantvillage_snn")
!rm -rf {CODE}
!git clone --depth 1 --branch {BRANCH} {GITHUB_URL} {CODE}

os.chdir(CODE)
sys.path.insert(0, str(CODE))
print("CWD", os.getcwd())

In [ ]:
import torch
from dataclasses import fields, replace

from config import TrainConfig
from data.kaggle_dataloader import get_kaggle_dataloaders
from models.res2net_msf import build_model
from models.spiking_resnet18_backbone import build_spiking_resnet18
from train import encode_batch
from utils.seed import set_seed
from utils.visualizer import (
    save_cross_layer_spike_heatmap,
    save_cross_layer_spike_raster,
    save_layer_mean_spike_bar,
    save_paper_panel,
    visualize_sample,
)

CKPT_PATH = Path("/kaggle/working/best_msf_res2net.pt")
DATA_ROOT = Path("/kaggle/input") / "your-plantvillage-dataset"
OUT_DIR = Path("/kaggle/working/paper_snn_viz")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 与训练 checkpoint 一致：Res2Net SNN 或 SpikingJelly ResNet-18 SNN
BACKBONE = "res2net"  # "res2net" | "sj_resnet18"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
d = ckpt["cfg"]
names = {f.name for f in fields(TrainConfig)}
cfg = replace(TrainConfig(), **{k: v for k, v in d.items() if k in names})
set_seed(cfg.seed)

_, val_loader, nc = get_kaggle_dataloaders(
    str(DATA_ROOT),
    image_size=cfg.image_size,
    batch_size=4,
    num_workers=2,
    val_ratio=0.2,
    seed=cfg.seed,
    auto_find_subdir=True,
)
cfg.num_classes = nc
if BACKBONE == "sj_resnet18":
    model = build_spiking_resnet18(cfg).to(device)
else:
    model = build_model(cfg).to(device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

images, _labels = next(iter(val_loader))
images = images[:1].to(device)
x = encode_batch(images, cfg.T, device)
print("spike tensor", tuple(x.shape), "T=", cfg.T)

In [ ]:
save_cross_layer_spike_heatmap(
    model,
    x,
    str(OUT_DIR / "paper_snn_layer_time_heatmap.png"),
    title=f"Cross-layer spike activity vs time ($T={cfg.T}$)",
    max_rows=32,
    dpi=300,
)
save_cross_layer_spike_raster(
    model,
    x,
    str(OUT_DIR / "paper_snn_spike_raster.png"),
    title=f"Subsampled spike raster (real forward, $T={cfg.T}$)",
    max_layers=14,
    neurons_per_layer=40,
    seed=0,
    dpi=300,
)
save_paper_panel(
    model,
    x,
    str(OUT_DIR / "paper_snn_panel.png"),
    title=f"MSF-Res2Net ($T={cfg.T}$, trained checkpoint)",
    dpi=300,
)
save_layer_mean_spike_bar(
    model,
    x,
    str(OUT_DIR / "paper_snn_layer_spike_rates.png"),
    max_bars=20,
)
visualize_sample(model, x, str(OUT_DIR), tag="paper")
print("Saved under", OUT_DIR)